# Tuần 3 (Sprint S1) — Huấn luyện DistilBERT E5, Đóng băng Encoder E7 & Sinh Nhãn Mềm LLM P2 (TV3 — Tasks 3.9, 3.10, 3.11)

Notebook này thực thi các thí nghiệm GPU Tuần 3 của **TV3** trên Google Colab (`Tesla T4`) theo `docs/WEEK3.md`:
1. **Task 3.9 (`E5` — DistilBERT fine-tune toàn bộ, 3 seeds `[1, 2, 3]`):**
   - Chạy `python -m models.transformer --config configs/E5.yaml`.
   - Đọc split cố định `data/splits/cluster_split.jsonl` (`SHA-256: 8c2c35ea...`) qua `train.data.load_split_data`.
   - Xuất `results/E5/seed{1,2,3}/{run.json, preds_validation.jsonl}` và chấm điểm qua `evaluate.metrics`.
2. **Task 3.10 (`E7` — DistilBERT đóng băng encoder, chỉ huấn luyện classification head, 3 seeds `[1, 2, 3]`):**
   - Chạy `python -m models.transformer --config configs/E7.yaml` (copy từ `E5.yaml`, chỉ đổi `freeze_encoder: true`).
   - Xuất `results/E7/seed{1,2,3}/{run.json, preds_validation.jsonl}` và chấm điểm qua `evaluate.metrics`.
3. **Task 3.11 (Sinh nhãn mềm LLM với prompt `P2` trên kho `train_unlabeled` năm 2026):**
   - Chạy `python -m models.llm_soft_labels --headlines data/raw/headlines_2026/headlines.jsonl`.
   - Tự động lọc chỉ giữ `pool == "train_unlabeled"` và loại trừ tuyệt đối mọi ID trong `labeling/trial30_ids.csv` và `labeling/test300_ids.csv`.
   - Xuất `results/llm_soft_labels/p2_soft_labels.jsonl` (chỉ gồm ID và xác suất/log-prob, không chứa văn bản tiêu đề) và `results/llm_soft_labels/run.json`.

## 1. Cài đặt Môi trường Colab, Clone Nhánh `tv3/week3` & Tải Dữ liệu Thô FNS

- Nạp `HF_TOKEN` và `GH_TOKEN` qua **Colab Secrets** (`google.colab.userdata`) với HTTP header tạm thời.
- Chạy `python data/download.py --only fns` để tạo `data/raw/fns/{train,validation}.jsonl`.

In [ ]:
import base64
import os
import platform
import subprocess
import sys
from pathlib import Path
from google.colab import userdata

token_val = userdata.get("HF_TOKEN")
if not token_val:
    raise RuntimeError("Missing HF_TOKEN in Colab Secrets.")
os.environ["HF_TOKEN"] = token_val.strip()
del token_val

REPO_URL = "https://github.com/thienandinh29/financial-sentiment-dl.git"
REPO_DIR = Path("financial-sentiment-dl")
BRANCH = "tv3/week3"

if not REPO_DIR.exists():
    gh_val = userdata.get("GH_TOKEN")
    if not gh_val:
        raise RuntimeError("Missing GH_TOKEN in Colab Secrets.")
    basic_auth = base64.b64encode(f"x-access-token:{gh_val.strip()}".encode("utf-8")).decode("ascii")
    git_env = os.environ.copy()
    git_env.update({
        "GIT_TERMINAL_PROMPT": "0",
        "GIT_CONFIG_COUNT": "1",
        "GIT_CONFIG_KEY_0": "http.https://github.com/.extraheader",
        "GIT_CONFIG_VALUE_0": f"AUTHORIZATION: basic {basic_auth}",
    })
    del gh_val, basic_auth
    res = subprocess.run(
        ["git", "clone", "-q", "-b", BRANCH, REPO_URL, str(REPO_DIR)],
        env=git_env,
        capture_output=True,
        text=True,
        check=False,
    )
    del git_env
    if res.returncode != 0:
        raise RuntimeError(f"git clone failed with exit code {res.returncode}.")

if Path.cwd().name != REPO_DIR.name and REPO_DIR.exists():
    os.chdir(REPO_DIR)

remote_origin = subprocess.run(
    ["git", "config", "--get", "remote.origin.url"],
    capture_output=True,
    text=True,
    check=True,
).stdout.strip()
assert remote_origin == REPO_URL, "Unexpected remote.origin.url format."

py_ver = platform.python_version()
print(f"Python version: {py_ver}")
if Path("requirements.lock").is_file():
    lock_proc = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.lock"],
        capture_output=True,
        text=True,
        check=False,
    )
    if lock_proc.returncode == 0:
        print("Installed dependencies from requirements.lock successfully.")
    else:
        print(f"requirements.lock failed on Python {py_ver}; falling back to requirements.txt.")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

if not (Path("data/raw/fns/train.jsonl").is_file() and Path("data/raw/fns/validation.jsonl").is_file()):
    subprocess.run([sys.executable, "data/download.py", "--only", "fns"], check=True)

## 2. Task 3.9 — Chạy `E5` (DistilBERT Fine-Tune Toàn Bộ, 3 Seeds)

- Huấn luyện `distilbert/distilbert-base-uncased` với `freeze_encoder: false` trên 3 seeds (`[1, 2, 3]`).
- Lưu kết quả vào `results/E5/seed{1,2,3}/{run.json, preds_validation.jsonl}` và kiểm tra qua `evaluate.metrics`.

In [ ]:
import json
import numpy as np
import yaml
from pathlib import Path
from evaluate.metrics import read_preds, score, validation_gold
from models.transformer import run_transformer_experiment

e5_cfg = yaml.safe_load(Path("configs/E5.yaml").read_text(encoding="utf-8"))
e5_summaries = run_transformer_experiment(e5_cfg)
gold = validation_gold(e5_cfg)

e5_f1s = []
for s in e5_summaries:
    seed = s["seed"]
    pred_path = Path(f"results/E5/seed{seed}/preds_validation.jsonl")
    scored = score(read_preds(pred_path), gold)
    e5_f1s.append(scored["macro_f1"])
    print(f"E5 seed {seed}: best_epoch={s['best_epoch']} | val_macro_f1={scored['macro_f1']:.4f} | f1={scored['f1']}")

print(f"E5 Validation Macro-F1 (3 seeds): {np.mean(e5_f1s):.4f} +/- {np.std(e5_f1s, ddof=1):.4f}")

## 3. Task 3.10 — Chạy `E7` (DistilBERT Đóng Băng Encoder, 3 Seeds)

- Copy cấu hình từ `E5.yaml` và chỉ đổi một yếu tố duy nhất: `freeze_encoder: true`.
- Lưu kết quả vào `results/E7/seed{1,2,3}/{run.json, preds_validation.jsonl}` và kiểm tra qua `evaluate.metrics`.

In [ ]:
import json
import numpy as np
import yaml
from pathlib import Path
from evaluate.metrics import read_preds, score, validation_gold
from models.transformer import run_transformer_experiment

e7_cfg = yaml.safe_load(Path("configs/E7.yaml").read_text(encoding="utf-8"))
e7_summaries = run_transformer_experiment(e7_cfg)
gold = validation_gold(e7_cfg)

e7_f1s = []
for s in e7_summaries:
    seed = s["seed"]
    pred_path = Path(f"results/E7/seed{seed}/preds_validation.jsonl")
    scored = score(read_preds(pred_path), gold)
    e7_f1s.append(scored["macro_f1"])
    print(f"E7 seed {seed}: trainable_params={s['trainable_params']}/{s['total_params']} | best_epoch={s['best_epoch']} | val_macro_f1={scored['macro_f1']:.4f} | f1={scored['f1']}")

print(f"E7 Validation Macro-F1 (3 seeds): {np.mean(e7_f1s):.4f} +/- {np.std(e7_f1s, ddof=1):.4f}")

## 4. Task 3.11 — Sinh Nhãn Mềm LLM với Prompt `P2` trên Kho `train_unlabeled` Năm 2026

- Yêu cầu có file `data/raw/headlines_2026/headlines.jsonl` (do TV2 xuất từ `scripts/export_headlines.py`, không commit lên Git).
- Chỉ chấm các dòng có `pool == "train_unlabeled"` và loại trừ toàn bộ ID trong `labeling/trial30_ids.csv` và `labeling/test300_ids.csv`.
- Hỗ trợ chạy tiếp (resume) nếu phiên Colab ngắt giữa chừng; lưu kết quả không chứa văn bản tiêu đề vào `results/llm_soft_labels/p2_soft_labels.jsonl` và `results/llm_soft_labels/run.json`.

In [ ]:
import gc
import json
from pathlib import Path
import torch
from models.llm_soft_labels import DEFAULT_HEADLINES, generate_soft_labels

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

if Path(DEFAULT_HEADLINES).is_file():
    soft_summary = generate_soft_labels(headlines_path=DEFAULT_HEADLINES)
    print(json.dumps(soft_summary, indent=2))
else:
    print(f"Skipping Task 3.11 until {DEFAULT_HEADLINES} is uploaded to Colab runtime.")